In [1]:
!pip install wfdb tensorflowjs
import wfdb
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
import os

In [3]:
import wfdb
import numpy as np
import os

# Thông số chuẩn hóa và cấu hình
FS = 360 # [cite: 202, 360]
WINDOW_SIZE = 100 #
MEAN = -0.28766632142632625 #
STD = 0.5241760803999351 #

def load_data():
    all_heartbeats = []
    all_labels = []

    # Danh sách các bản ghi tiêu biểu từ MIT-BIH [cite: 200]
    records = ['100', '101', '103', '105', '106', '108', '111', '113', '115', '118', '119',
               '121', '122', '123', '124', '200', '201', '202', '203', '205', '208', '210']

    # SỬA LỖI TẠI ĐÂY: dùng 'mitdb' thay vì 'mitbih'
    db_path = 'mit-bih-arrhythmia-database-1.0.0'
    if not os.path.exists(db_path):
        wfdb.dl_database('mitdb', db_path)

    for res in records:
        path = os.path.join(db_path, res)
        record = wfdb.rdrecord(path)
        ann = wfdb.rdann(path, 'atr')

        # Trích xuất tín hiệu Lead II (thường là MLII) [cite: 201, 203, 205]
        signals = record.p_signal
        lead_ii_idx = 0 if record.sig_name[0] == 'MLII' else 1
        signal = signals[:, lead_ii_idx]

        # Chuẩn hóa tín hiệu (Scaling)
        signal = (signal - MEAN) / STD

        # Bản đồ nhãn theo tiêu chuẩn AAMI (N, S, V, F) [cite: 199, 233]
        label_map = {'N':0, 'L':0, 'R':0, 'e':0, 'j':0,
                     'A':1, 'a':1, 'J':1, 'S':1,
                     'V':2, 'E':2,
                     'F':3}

        for i in range(len(ann.sample)):
            sym = ann.symbol[i]
            if sym in label_map:
                idx = ann.sample[i]
                # Cắt cửa sổ 100 mẫu quanh đỉnh R
                if idx - WINDOW_SIZE//2 >= 0 and idx + WINDOW_SIZE//2 < len(signal):
                    beat = signal[idx - WINDOW_SIZE//2 : idx + WINDOW_SIZE//2]
                    all_heartbeats.append(beat)
                    all_labels.append(label_map[sym])

    return np.array(all_heartbeats).reshape(-1, WINDOW_SIZE, 1), np.array(all_labels)

X, y = load_data()

Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record list for: 111
Generating record list for: 112
Generating record list for: 113
Generating record list for: 114
Generating record list for: 115
Generating record list for: 116
Generating record list for: 117
Generating record list for: 118
Generating record list for: 119
Generating record list for: 121
Generating record list for: 122
Generating record list for: 123
Generating record list for: 124
Generating record list for: 200
Generating record list for: 201
Generating record list for: 202
Generating record list for: 203
Generating record list for: 205
Generating record list for: 207
Generating record list for: 208
Generating record list for: 209
Generati

In [15]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_model():
    model = models.Sequential()

    # Sử dụng Input(shape) thay vì input_shape trong Conv1D để tránh cảnh báo
    # WINDOW_SIZE = 100 theo tệp model.json của bạn
    model.add(layers.Input(shape=(WINDOW_SIZE, 1)))

    # Block 1: Trích xuất đặc trưng cơ bản [cite: 125, 313]
    model.add(layers.Conv1D(128, 10, activation='relu', padding='same')) # [cite: 315, 316]
    model.add(layers.Conv1D(128, 10, activation='relu', padding='same'))
    model.add(layers.MaxPooling1D(2)) # [cite: 314, 318]
    model.add(layers.Dropout(0.2)) # [cite: 321]
    model.add(layers.BatchNormalization()) #

    # Block 2: Trích xuất đặc trưng bậc trung [cite: 125, 313, 630]
    model.add(layers.Conv1D(128, 10, activation='relu', padding='same'))
    model.add(layers.Conv1D(128, 10, activation='relu', padding='same'))
    model.add(layers.MaxPooling1D(2))
    model.add(layers.Dropout(0.2))
    model.add(layers.BatchNormalization())

    # Block 3: Trích xuất đặc trưng phức tạp [cite: 125, 313, 630]
    model.add(layers.Conv1D(128, 10, activation='relu', padding='same'))
    model.add(layers.Conv1D(128, 10, activation='relu', padding='same'))
    model.add(layers.MaxPooling1D(2))
    model.add(layers.Dropout(0.2))
    model.add(layers.BatchNormalization())

    # Fully Connected Layers: Phân loại nhịp tim [cite: 255, 331]
    model.add(layers.Flatten()) # [cite: 332]
    model.add(layers.Dense(512, activation='relu')) # [cite: 335]
    model.add(layers.Dropout(0.2))
    model.add(layers.Dense(4, activation='softmax')) # Phân loại 4 nhóm N, S, V, F [cite: 336]

    # Cấu hình Optimizer: Loại bỏ tham số 'decay' lỗi thời và thay bằng cấu trúc mới [cite: 326, 337]
    # 'decay' trong bài báo cũ tương ứng với 'weight_decay' hoặc xử lý qua learning_rate schedule
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_model()
model.summary() # Kiểm tra lại kiến trúc mô hình

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_30 (Conv1D)              │ (None, 100, 128)       │         1,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_31 (Conv1D)              │ (None, 100, 128)       │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_15 (MaxPooling1D) │ (None, 50, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 50, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 50, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_32 (Conv1D)              │ (None, 50, 128)        │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_33 (Conv1D)              │ (None, 50, 128)        │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_16 (MaxPooling1D) │ (None, 25, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 25, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 25, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_34 (Conv1D)              │ (None, 25, 128)        │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_35 (Conv1D)              │ (None, 25, 128)        │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_17 (MaxPooling1D) │ (None, 12, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_22 (Dropout)            │ (None, 12, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 12, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 512)            │       786,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_23 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 4)              │         2,052 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,611,780 (6.15 MB)

 Trainable params: 1,611,012 (6.15 MB)

 Non-trainable params: 768 (3.00 KB)

In [16]:
from sklearn.model_selection import train_test_split

# Chia dữ liệu: 75% cho huấn luyện & validation, 25% cho kiểm tra (Testing) [cite: 243, 380]
# Sử dụng 'stratify=y' để đảm bảo tỷ lệ các lớp N, S, V, F đồng đều [cite: 244, 245]
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Chia tiếp tập train_full thành Train (80%) và Validation (20%) [cite: 246]
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
)

print(f"Số lượng mẫu huấn luyện (y_train): {len(y_train)}")

Số lượng mẫu huấn luyện (y_train): 28872


In [17]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# 1. Tính toán class weight (Giữ nguyên vì bạn đã làm đúng)
weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(weights))

# 2. Thêm các cơ chế giám sát thông minh
callbacks = [
    # Dừng train nếu val_loss không giảm sau 20 epoch để tránh quá nhiệt
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),

    # Tự động giảm Learning Rate đi 2 lần nếu val_loss bị "đứng hình" sau 10 epoch
    # Giúp mô hình hội tụ sâu hơn và giảm dao động
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=0.00001, verbose=1),

    # Lưu lại model tốt nhất (dùng cho Server Node.js sau này)
    ModelCheckpoint('best_ecg_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
]

# 3. Huấn luyện mô hình với cấu hình tối ưu [cite: 276, 338]
history = model.fit(
    X_train, y_train,
    epochs=500, # Vẫn để 500, EarlyStopping sẽ tự dừng khi đạt đỉnh [cite: 338]
    batch_size=512, # [cite: 326]
    validation_data=(X_val, y_val), # Dùng tập val đã chia sẵn thay vì split lại
    class_weight=class_weights, # Xử lý mất cân bằng lớp [cite: 249]
    callbacks=callbacks, # Thêm bộ giám sát vừa tạo
    verbose=1
)

# 4. Đánh giá cuối cùng trên tập Test độc lập [cite: 85, 341]
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"\n---> Độ chính xác cuối cùng trên tập Test: {test_acc*100:.2f}%")

Epoch 1/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step - accuracy: 0.5122 - loss: 1.2834
Epoch 1: val_accuracy improved from -inf to 0.66487, saving model to best_ecg_model.h5


57/57 ━━━━━━━━━━━━━━━━━━━━ 20s 176ms/step - accuracy: 0.5132 - loss: 1.2788 - val_accuracy: 0.6649 - val_loss: 1.1902 - learning_rate: 0.0010
Epoch 2/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6401 - loss: 0.6934
Epoch 2: val_accuracy improved from 0.66487 to 0.93419, saving model to best_ecg_model.h5


57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 0.6400 - loss: 0.6931 - val_accuracy: 0.9342 - val_loss: 0.4034 - learning_rate: 0.0010
Epoch 3/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.7518 - loss: 0.5594
Epoch 3: val_accuracy did not improve from 0.93419
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.7508 - loss: 0.5595 - val_accuracy: 0.4612 - val_loss: 1.0991 - learning_rate: 0.0010
Epoch 4/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6802 - loss: 0.4964
Epoch 4: val_accuracy did not improve from 0.93419
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.6805 - loss: 0.4964 - val_accuracy: 0.6852 - val_loss: 0.6412 - learning_rate: 0.0010
Epoch 5/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.6507 - loss: 0.4440
Epoch 5: val_accuracy did not improve from 0.93419
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 0.6514 - loss: 0.4442 - val_accuracy: 0.7891 - val_loss: 0.5097 - learning_rate: 0.0010
Epoch 6/500
57/57 ━━━━━━━━━━━

57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 48ms/step - accuracy: 0.9398 - loss: 0.0786 - val_accuracy: 0.9467 - val_loss: 0.1366 - learning_rate: 2.5000e-04
Epoch 42/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9354 - loss: 0.0783
Epoch 42: val_accuracy did not improve from 0.94666
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.9354 - loss: 0.0783 - val_accuracy: 0.9331 - val_loss: 0.1529 - learning_rate: 2.5000e-04
Epoch 43/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9342 - loss: 0.0793
Epoch 43: val_accuracy improved from 0.94666 to 0.94957, saving model to best_ecg_model.h5


57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 0.9342 - loss: 0.0792 - val_accuracy: 0.9496 - val_loss: 0.1357 - learning_rate: 2.5000e-04
Epoch 44/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9424 - loss: 0.0733
Epoch 44: val_accuracy did not improve from 0.94957
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.9424 - loss: 0.0733 - val_accuracy: 0.9228 - val_loss: 0.1791 - learning_rate: 2.5000e-04
Epoch 45/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9364 - loss: 0.0646
Epoch 45: val_accuracy improved from 0.94957 to 0.95664, saving model to best_ecg_model.h5


...

57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 0.9828 - loss: 0.0193 - val_accuracy: 0.9796 - val_loss: 0.0714 - learning_rate: 3.1250e-05
Epoch 95/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.9839 - loss: 0.0221
Epoch 95: val_accuracy improved from 0.97963 to 0.98005, saving model to best_ecg_model.h5


57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step - accuracy: 0.9839 - loss: 0.0221 - val_accuracy: 0.9800 - val_loss: 0.0721 - learning_rate: 3.1250e-05
Epoch 96/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9864 - loss: 0.0191
Epoch 96: val_accuracy did not improve from 0.98005
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.9864 - loss: 0.0191 - val_accuracy: 0.9770 - val_loss: 0.0786 - learning_rate: 3.1250e-05
Epoch 97/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.9843 - loss: 0.0186
Epoch 97: val_accuracy did not improve from 0.98005
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.9844 - loss: 0.0186 - val_accuracy: 0.9752 - val_loss: 0.0791 - learning_rate: 3.1250e-05
Epoch 98/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.9835 - loss: 0.0199
Epoch 98: val_accuracy did not improve from 0.98005
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.9835 - loss: 0.0199 - val_accuracy: 0.9776 - val_loss: 0.0752 - learning_rate: 3.1250e-05
Epoch 9

57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.9865 - loss: 0.0169 - val_accuracy: 0.9810 - val_loss: 0.0694 - learning_rate: 3.1250e-05
Epoch 100/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.9870 - loss: 0.0164
Epoch 100: val_accuracy did not improve from 0.98102
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.9870 - loss: 0.0164 - val_accuracy: 0.9730 - val_loss: 0.0888 - learning_rate: 3.1250e-05
Epoch 101/500
56/57 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.9836 - loss: 0.0218
Epoch 101: val_accuracy did not improve from 0.98102
57/57 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - accuracy: 0.9837 - loss: 0.0218 - val_accuracy: 0.9742 - val_loss: 0.0849 - learning_rate: 3.1250e-05
Epoch 102/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9839 - loss: 0.0181
Epoch 102: val_accuracy did not improve from 0.98102
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.9839 - loss: 0.0181 - val_accuracy: 0.9796 - val_loss: 0.0706 - learning_rate: 3.1250e-05
E

57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 48ms/step - accuracy: 0.9865 - loss: 0.0180 - val_accuracy: 0.9824 - val_loss: 0.0677 - learning_rate: 3.1250e-05
Epoch 104/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9872 - loss: 0.0223
Epoch 104: val_accuracy did not improve from 0.98241
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.9872 - loss: 0.0223 - val_accuracy: 0.9821 - val_loss: 0.0690 - learning_rate: 3.1250e-05
Epoch 105/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.9879 - loss: 0.0159
Epoch 105: val_accuracy did not improve from 0.98241
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.9879 - loss: 0.0159 - val_accuracy: 0.9798 - val_loss: 0.0707 - learning_rate: 3.1250e-05
Epoch 106/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9871 - loss: 0.0164
Epoch 106: val_accuracy did not improve from 0.98241
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.9871 - loss: 0.0165 - val_accuracy: 0.9787 - val_loss: 0.0750 - learning_rate: 3.1250e-05
E

57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 0.9891 - loss: 0.0131 - val_accuracy: 0.9842 - val_loss: 0.0656 - learning_rate: 1.5625e-05
Epoch 121/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.9893 - loss: 0.0151
Epoch 121: val_accuracy did not improve from 0.98421
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.9893 - loss: 0.0151 - val_accuracy: 0.9817 - val_loss: 0.0683 - learning_rate: 1.5625e-05
Epoch 122/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.9896 - loss: 0.0123
Epoch 122: val_accuracy did not improve from 0.98421
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 0.9896 - loss: 0.0123 - val_accuracy: 0.9831 - val_loss: 0.0668 - learning_rate: 1.5625e-05
Epoch 123/500
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.9890 - loss: 0.0147
Epoch 123: val_accuracy did not improve from 0.98421
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.9890 - loss: 0.0147 - val_accuracy: 0.9794 - val_loss: 0.0745 - learning_rate: 1.5625e-05
E

In [18]:
import tensorflowjs as tfjs

# Lưu mô hình định dạng Keras (.h5)
model.save("ecg_model_final.h5")

# Chuyển đổi sang định dạng TFJS cho Node.js
tfjs.converters.save_keras_model(model, "tfjs_model_output")

print("Đã xuất mô hình thành công! Hãy tải thư mục 'tfjs_model_output' về Server.")

failed to lookup keras version from the file,
    this is likely a weight only file
Đã xuất mô hình thành công! Hãy tải thư mục 'tfjs_model_output' về Server.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')